In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/small_bench/checkpoints/pre_cell_10.pickle

In [3]:
%%RecordEvent
%%time
### cell 10 ###

# Local mappings for clean category names
_map_2017 = {
    'Male': 'Man',
    'Female': 'Woman',
    'A different identity': 'Prefer to self-describe',
    'Non-binary, genderqueer, or gender non-conforming': 'Prefer to self-describe',
    'Nonbinary': 'Prefer to self-describe',
    'Prefer not to say': 'Prefer to self-describe'
}
_map_other = {
    'Male': 'Man',
    'Female': 'Woman',
    'Nonbinary': 'Prefer to self-describe',
    'Prefer not to say': 'Prefer to self-describe'
}
# Source DataFrame mapping
def _build_df_map():
    # delayed lookup to avoid modifying future vars
    return {
        '2017': responses_df_2017,
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
_df_map = _build_df_map()

def count_then_return_percent_22(df, col):
    """
    Return percentage distribution for df[col], including NaNs in counts,
    relative to non-null entries, rounded to one decimal.
    """
    return (df[col]
            .value_counts(dropna=False)
            .div(df[col].count())
            .mul(100)
            .round(1))

def combine_subset_of_data_from_multiple_years_22(question, x_axis_title, include_2017=None):
    """
    For each year in the selected range, map and count percentages of the given question,
    then concatenate into a single DataFrame with columns ['percentage', 'year', x_axis_title].
    """
    other_years = ['2018', '2019', '2020', '2021', '2022']
    years = (['2017'] + other_years) if include_2017 is not None else other_years

    # Compute percentage series for each year and concatenate in one go
    pct_series = {
        year: (
            _df_map[year][('GenderSelect' if year == '2017' else question)]
            .replace(_map_2017 if year == '2017' else _map_other)
            .value_counts(dropna=False, normalize=True)
            .mul(100)
            .round(1)
        )
        for year in years
    }
    df_combined = (
        pd.concat(pct_series, names=['year', x_axis_title])
          .reset_index(name='percentage')
    )
    return df_combined[['percentage', 'year', x_axis_title]]

# Usage remains unchanged
question_of_interest_cell22 = 'What is your gender? - Selected Choice'
title_for_x_axis_cell22 = ''
age_df_combined_cell22 = combine_subset_of_data_from_multiple_years_22(
    question_of_interest_cell22,
    title_for_x_axis_cell22,
    include_2017='yes'
)
age_df_combined_cell22 = age_df_combined_cell22.sort_values(
    by=['year', 'percentage'],
    ascending=True
)
age_df_combined_cell22.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19 entries, 3 to 16
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   percentage  19 non-null     float64
 1   year        19 non-null     object 
 2               18 non-null     object 
dtypes: float64(1), object(2)
memory usage: 608.0+ bytes
CPU times: user 38.4 ms, sys: 0 ns, total: 38.4 ms
Wall time: 38.3 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_10_try_2.pickle